# 03 — Avaliação

Aplica a régua de critérios (versão 1.0) às 15 políticas de privacidade com um modelo de linguagem e calcula a nota de cada uma.

A lógica está em `src/avaliacao.py`. Este notebook executa as etapas e mostra os resultados, na ordem:

1. como o modelo é chamado;
2. avaliação das políticas;
3. conferência das evidências;
4. estabilidade do modelo entre duas rodadas;
5. validação contra a avaliação manual (padrão-ouro);
6. notas e respostas por critério.

In [1]:
from pathlib import Path
import os
import sys

# Os caminhos do projeto são relativos à raiz do repositório.
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_colwidth", 100)

## 1. Como o modelo é chamado

Todas as políticas recebem o **mesmo prompt**: o texto da régua (definições, regras de resposta e os 12 critérios, sem os pesos) seguido dos documentos da política. Só os documentos mudam.

O modelo responde em JSON, em um formato fixo (`prompts/avaliacao_esquema.json`): uma resposta por critério, com o trecho da política que a sustenta, mais os fatos declarados — tipos de dado, terceiros e países.

In [2]:
from src.avaliacao import ESFORCO, MODELO, VERSAO_PROMPT, VERSAO_REGUA, montar_prompt

sistema, usuario = montar_prompt("P10")
print(f"modelo: {MODELO} | esforço: {ESFORCO} | prompt: {VERSAO_PROMPT} | régua: {VERSAO_REGUA}")
print(f"prompt de sistema: {len(sistema)} caracteres | prompt de usuário (P10): {len(usuario)} caracteres")
print()
print(sistema)

modelo: claude-opus-5-5 | esforço: high | prompt: v1 | régua: 1.0
prompt de sistema: 1107 caracteres | prompt de usuário (P10): 17217 caracteres

Você avalia políticas de privacidade de fabricantes de aparelhos de EEG de consumo. Sua tarefa é aplicar uma régua de critérios ao texto de uma política e registrar o que o texto declara.

Regras gerais:

- Avalie apenas o que está escrito nos documentos fornecidos. Não use conhecimento externo sobre a empresa, seus produtos ou suas práticas.
- Você descreve o texto; não julga a empresa nem afirma que ela cumpre ou descumpre alguma lei.
- Siga a régua à risca, inclusive as regras de decisão de cada critério e as regras gerais sobre as respostas.
- Toda evidência é um trecho literal dos documentos: copie-o sem alterar nenhuma palavra, no idioma original, sem traduzir, sem resumir e sem juntar trechos distantes. Use um único trecho contínuo de até 400 caracteres.
- Quando a resposta for `nao_menciona`, deixe a evidência e o identificador do doc

O modelo é acionado pelo Claude Code em modo não interativo, sem ferramentas, sem as configurações do usuário e a partir de uma pasta vazia: a classificação depende só do prompt.

## 2. Avaliação das políticas

Cada resposta bruta fica em `data/interim/avaliacao/llm/`. Políticas já avaliadas são puladas, então a célula abaixo não repete chamadas.

In [3]:
from src.avaliacao import avaliar_politica

politicas = sorted(pd.read_csv("data/raw/fontes_politicas.csv")["id_politica"].unique())
arquivos = [avaliar_politica(politica) for politica in politicas]
print(f"{len(arquivos)} políticas avaliadas")

15 políticas avaliadas


In [4]:
from src.avaliacao import gravar_tabelas

tabelas = gravar_tabelas()
avaliacoes = pd.DataFrame(tabelas["avaliacoes"])
pontuacao = pd.DataFrame(tabelas["pontuacao"])
{nome: len(linhas) for nome, linhas in tabelas.items()}

{'avaliacoes': 180,
 'dados': 210,
 'terceiros': 175,
 'paises': 21,
 'pontuacao': 15}

## 3. Conferência das evidências

A régua exige um trecho da política para toda resposta `sim` ou `nao`. Cada trecho citado pelo modelo é procurado, palavra por palavra, no texto limpo da política. Uma resposta `sim` ou `nao` cujo trecho não é encontrado fica marcada para revisão.

In [5]:
pd.crosstab(avaliacoes["resposta"], avaliacoes["evidencia_conferida"], margins=True, margins_name="total")

evidencia_conferida,confere,nao_confere,sem_evidencia,total
resposta,,,,
nao,25,0,0,25
nao_menciona,23,2,65,90
sim,65,0,0,65
total,113,2,65,180


In [6]:
revisar = avaliacoes[avaliacoes["precisa_revisao"] == "sim"]
print(f"{len(revisar)} respostas marcadas para revisão")
revisar[["id_politica", "id_criterio", "resposta", "evidencia_conferida", "evidencia"]]

0 respostas marcadas para revisão


,id_politica,id_criterio,resposta,evidencia_conferida,evidencia


In [7]:
fatos = {nome: pd.DataFrame(tabelas[nome]) for nome in ("dados", "terceiros", "paises")}
pd.DataFrame({nome: tabela["evidencia_conferida"].value_counts() for nome, tabela in fatos.items()}).fillna(0).astype(int)

,dados,terceiros,paises
evidencia_conferida,,,
confere,210,175,21


## 4. Estabilidade do modelo

O modelo não é determinístico: a mesma política pode receber respostas diferentes em duas execuções. Para medir isso, as três políticas da amostra do padrão-ouro foram avaliadas uma segunda vez, com o mesmo prompt.

In [8]:
from src.avaliacao import concordancia, consolidar

amostra = pd.read_csv("data/interim/avaliacao/amostra_padrao_ouro.csv")
rodada_2 = pd.DataFrame(consolidar(rodada=2)["avaliacoes"])
pares = avaliacoes.merge(rodada_2, on=["id_politica", "id_criterio"], suffixes=("_r1", "_r2"))
print("políticas repetidas:", ", ".join(sorted(pares["id_politica"].unique())))
concordancia(list(pares["resposta_r1"]), list(pares["resposta_r2"]))

políticas repetidas: P01, P05, P13


{'itens': 36, 'concordancia': 0.972, 'kappa': 0.953}

In [9]:
pares.loc[pares["resposta_r1"] != pares["resposta_r2"], ["id_politica", "id_criterio", "resposta_r1", "resposta_r2", "justificativa_r1", "justificativa_r2"]]

,id_politica,id_criterio,resposta_r1,resposta_r2,justificativa_r1,justificativa_r2
28,P13,C05,nao_menciona,sim,"A política fala em acompanhar a duração do sono e a atividade cerebral, mas não separa o sinal b...","A política nomeia uma informação derivada dos dados biométricos, a duração do sono a cada noite...."


## 5. Validação contra o padrão-ouro

O padrão-ouro é a avaliação manual de três políticas, sorteadas com semente fixa antes de qualquer resultado do modelo (`src/amostra_padrao_ouro.py`): duas do grupo principal e uma do grupo de comparação.

A planilha a preencher é `data/interim/avaliacao/manual/padrao_ouro.csv`. Quem a preenche não deve consultar antes as respostas do modelo.

In [10]:
amostra

,id_politica,fabricante,grupo,semente,data_sorteio
0,P01,InteraXon Inc.,brasil,20261115,2026-11-15
1,P05,BrainBit Inc.,brasil,20261115,2026-11-15
2,P13,StimScience Inc.,comparacao,20261115,2026-11-15


In [11]:
manual = pd.read_csv("data/interim/avaliacao/manual/padrao_ouro.csv", dtype=str, keep_default_na=False)
preenchidas = manual[manual["resposta"] != ""]
print(f"{len(preenchidas)} de {len(manual)} respostas manuais preenchidas")

if len(preenchidas) == len(manual):
    comparacao = manual.merge(avaliacoes, on=["id_politica", "id_criterio"], suffixes=("_manual", "_modelo"))
    print(concordancia(list(comparacao["resposta_manual"]), list(comparacao["resposta_modelo"])))
    display(pd.crosstab(comparacao["resposta_manual"], comparacao["resposta_modelo"], margins=True, margins_name="total"))
else:
    print("Validação pendente: os resultados abaixo ainda não foram conferidos contra avaliação humana.")

0 de 36 respostas manuais preenchidas
Validação pendente: os resultados abaixo ainda não foram conferidos contra avaliação humana.


## 6. Resultados

**Atenção:** enquanto a validação da seção 5 estiver pendente, estes números são a leitura do modelo, não um resultado validado.

A nota ponderada soma os pesos dos critérios respondidos com `sim` (máximo de 25). A contagem simples de `sim` (máximo de 12) mostra se a ordem das políticas depende dos pesos.

In [12]:
pontuacao.sort_values("nota_ponderada", ascending=False)[["id_politica", "fabricante", "grupo", "nota_ponderada", "respostas_sim", "respostas_nao", "respostas_nao_menciona", "respostas_para_revisao"]]

,id_politica,fabricante,grupo,nota_ponderada,respostas_sim,respostas_nao,respostas_nao_menciona,respostas_para_revisao
11,P12,NextSense Inc.,comparacao,21,9,0,3,0
6,P07,Neurosity Inc.,brasil,16,8,1,3,0
2,P03,Emotiv Inc.,brasil,16,8,2,2,0
14,P15,Elemind Technologies Inc.,comparacao,16,7,2,3,0
9,P10,BrainCo Inc.,comparacao,14,7,1,4,0
0,P01,InteraXon Inc.,brasil,13,7,1,4,0
7,P08,Sens.ai Inc.,brasil,11,5,1,6,0
12,P13,StimScience Inc.,comparacao,8,3,4,5,0
10,P11,Neurable Inc.,comparacao,8,4,3,5,0
4,P05,BrainBit Inc.,brasil,5,2,0,10,0


In [13]:
pontuacao[["nota_ponderada", "respostas_sim"]] = pontuacao[["nota_ponderada", "respostas_sim"]].astype(int)
print("correlação de postos entre nota ponderada e contagem de sim:", round(pontuacao["nota_ponderada"].rank().corr(pontuacao["respostas_sim"].rank()), 3))
pontuacao.groupby("grupo")[["nota_ponderada", "respostas_sim"]].agg(["count", "median", "min", "max"])

correlação de postos entre nota ponderada e contagem de sim: 0.981


nota_ponderada                respostas_sim               
                    count median min max         count median min max
grupo                                                                
brasil                  9    5.0   0  16             9    2.0   0   8
comparacao              6   11.0   1  21             6    5.5   1   9

Respostas por critério, nos dois grupos. Silêncio (`nao_menciona`) e negativa (`nao`) valem os mesmos zero pontos, mas não significam a mesma coisa.

In [14]:
criterios = pd.read_csv("data/processed/criterios.csv")[["id", "criterio", "hipotese"]]
com_grupo = avaliacoes.merge(pontuacao[["id_politica", "grupo"]])
por_criterio = pd.crosstab([com_grupo["id_criterio"]], [com_grupo["grupo"], com_grupo["resposta"]])
criterios.merge(por_criterio.set_axis([f"{g}: {r}" for g, r in por_criterio.columns], axis=1), left_on="id", right_index=True)

,id,criterio,hipotese,brasil: nao,brasil: nao_menciona,brasil: sim,comparacao: nao,comparacao: nao_menciona,comparacao: sim
0,C01,Categoria específica,H1,0,3,6,0,1,5
1,C02,Proteção reforçada,H1,1,6,2,0,2,4
2,C03,Consentimento específico,H3,2,7,0,3,2,1
3,C04,Finalidades declaradas,H3,0,4,5,0,1,5
4,C05,Inferências informadas,H3,0,5,4,0,2,4
5,C06,Sem venda nem publicidade,H2,6,3,0,2,2,2
6,C07,Terceiros identificados,H2,0,6,3,0,2,4
7,C08,Local de armazenamento,H2,0,7,2,1,5,0
8,C09,Exclusão a pedido,H3,0,3,6,2,2,2
9,C10,Prazo de retenção,H3,0,5,4,0,4,2


## Limitações desta etapa

- As respostas vêm de um modelo de linguagem e ainda não foram validadas contra avaliação humana.
- A conferência automática garante que o trecho citado existe na política, não que ele sustenta a resposta.
- O modelo não é determinístico; a seção 4 mede o quanto as respostas variam.
- A régua (versão 1.0) ainda não passou por revisão jurídica.
- A avaliação descreve o texto das políticas na data da coleta, não a prática das empresas.